# 🧪 CAR Cytotoxicity Prediction Tool (Corrected Version)

This notebook uses the optimized XGBoost model to predict whether a new CAR construct will have 'High' or 'Low' cytotoxicity based on its protein sequences.

### Instructions
1.  **Run Cell 1** to install all required libraries.
2.  **Run Cell 2** and use the "Browse" button to upload the three saved files: `final_xgb_model.joblib`, `data_scaler.joblib`, and `model_columns.joblib`.
3.  **Run Cell 3** to load the models and functions. *This step may take a few minutes when loading the ESM-2 model for the first time.*
4.  **Choose a prediction method:**
    *   **Single Prediction:** Fill in the fields in **Cell 4** and run **Cells 5 and 6**.
    *   **Batch Prediction (Excel):** Run **Cell 7** directly.

### Step 1: Install Dependencies

In [1]:
# ==============================================================================
# Cell 1 (Corrected for Inference)
# ==============================================================================
# Added 'xlsxwriter' to the list of libraries to install
%pip install joblib scikit-learn pandas numpy torch transformers biopython xgboost tqdm xlsxwriter -q

print("✅ Required libraries, including the Excel engine (xlsxwriter), installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 12.9 MB/s eta 0:00:00
✅ Required libraries, including the Excel engine (xlsxwriter), installed.


### Step 2: Upload Model Files

Run the cell below and upload the 3 `.joblib` files when prompted.

In [2]:
# ==============================================================================
# Cell 2 (Complete and Corrected Version)
# ==============================================================================
from google.colab import files
import os

print("Please upload the 3 required files: 'final_xgb_model.joblib', 'data_scaler.joblib', and 'model_columns.joblib'")
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'File "{fn}" uploaded successfully.')

# Updated list of required files for the pipeline WITHOUT PCA
required_files = [
    'final_xgb_model.joblib',
    'data_scaler.joblib',
    'model_columns.joblib'
]

all_files_present = all(os.path.exists(f) for f in required_files)

if all_files_present:
    print("\n✅ All required files are present.")
else:
    print("\n❌ ERROR: One or more required files are missing. Please try again.")
    missing = [f for f in required_files if not os.path.exists(f)]
    print(f"Missing files: {missing}")

Please upload the 3 required files: 'final_xgb_model.joblib', 'data_scaler.joblib', and 'model_columns.joblib'


Saving data_scaler.joblib to data_scaler.joblib
Saving final_xgb_model.joblib to final_xgb_model.joblib
Saving model_columns.joblib to model_columns.joblib
File "data_scaler.joblib" uploaded successfully.
File "final_xgb_model.joblib" uploaded successfully.
File "model_columns.joblib" uploaded successfully.

✅ All required files are present.


### Step 3: Load Models and Define Functions

In [3]:
# ==============================================================================
# Cell 3 (Complete and Corrected Version): Load Models and Functions
# ==============================================================================
import pandas as pd
import numpy as np
import joblib
import torch
from transformers import EsmTokenizer, EsmModel
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import warnings

warnings.filterwarnings('ignore')

# --- Preprocessing Functions ---

def validate_and_clean_sequence(sequence, domain_name):
    """
    Validates that a sequence only contains valid amino acid characters and cleans it.
    Returns the cleaned sequence and a list of errors (empty if valid).
    """
    valid_aas = "ACDEFGHIKLMNPQRSTVWY*"
    errors = []

    if not isinstance(sequence, str) or not sequence:
        return "", errors # Returns an empty string for invalid inputs

    cleaned_seq = str(sequence).upper().replace(' ', '').strip()

    for i, char in enumerate(cleaned_seq):
        if char not in valid_aas:
            context_start = max(0, i - 10)
            context_end = min(len(cleaned_seq), i + 11)
            context = cleaned_seq[context_start:i] + f"->{char}<-" + cleaned_seq[i+1:context_end]
            error_msg = f"Domain '{domain_name}': Invalid character '{char}' found at position {i}. Context: ...{context}..."
            errors.append(error_msg)

    return cleaned_seq, errors

def calculate_prot_features(sequence):
    """Calculates the biophysical features."""
    default_features = {'length': 0, 'mol_weight': 0, 'pI': 0, 'aromaticity': 0, 'instability_idx': 0, 'gravy': 0}
    if pd.isna(sequence) or not sequence: return default_features
    sequence_cleaned = str(sequence).upper().strip().rstrip('*')
    if not sequence_cleaned:
        default_features['length'] = len(str(sequence))
        return default_features
    try:
        analysis = ProteinAnalysis(sequence_cleaned)
        return {
            'length': len(str(sequence)),
            'mol_weight': analysis.molecular_weight(),
            'pI': analysis.isoelectric_point(),
            'aromaticity': analysis.aromaticity(),
            'instability_idx': analysis.instability_index(),
            'gravy': analysis.gravy()
        }
    except Exception:
        return default_features

def get_single_embedding(sequence, model, tokenizer, device):
    """Generates the embedding for a single sequence."""
    if not isinstance(sequence, str) or len(sequence) == 0:
        return np.zeros(model.config.hidden_size)
    with torch.no_grad():
        inputs = tokenizer(sequence, return_tensors="pt", padding=True, truncation=True, max_length=1022).to(device)
        outputs = model(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        return embedding

# --- Loading Models ---
print("Loading saved objects...")
final_model = joblib.load('final_xgb_model.joblib')
scaler = joblib.load('data_scaler.joblib')
model_columns = joblib.load('model_columns.joblib')

print("Loading ESM-2 protein language model (this may take a few minutes)...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
esm_model_name = "facebook/esm2_t33_650M_UR50D"
tokenizer = EsmTokenizer.from_pretrained(esm_model_name)
esm_model = EsmModel.from_pretrained(esm_model_name).to(device)
esm_model.eval()

print(f"\n✅ Ready to make predictions on device: {device}")

Loading saved objects...
Loading ESM-2 protein language model (this may take a few minutes)...


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/534 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



✅ Ready to make predictions on device: cuda


---
### **Option A: Single Prediction**
---

In [ ]:
# ==============================================================================
# Cell 4a: CAR Domain Sequences
# ==============================================================================
#@title Sub-option 4a: CAR Domain Sequences
#@markdown Choose an example or enter your own sequences below.
example_to_load = 'Test Set Example (JAG1-F1)' #@param ["Validation Set Example (KM666_3XG4S)", "Test Set Example (JAG1-F1)"]

# Dictionary of example sequences
validation_example = {
    'peptide_signal_seq': 'METDTLLLWVLLLWPGSTG',
    'scfv_seq': 'QVQLQESGPGLVKPSQTLSICTVSGFSLASYNIIHWVRQPPGKLEWLGVIWAGGSTNYNSALMSRLTSIKDNSKNQVFLKMSSLTAADTAVYYCAKRSDDYSWFAYWGQGTLVTVSSGGGGSGGGGSGGGGSENQMTQSPSSLASVSGDRVTMTCRASSSVSSSYLHWYQQKSGKAPKWWIYSTSNLASGVPSRFGSGSGTDFTLTISSLQPEDFATTYCQQYSGYPITFGQGTKVEIKR',
    'hinge_seq': 'AEPKSPDKTHTCPPCPKDPK',
    'tm_seq': 'FWVLVVGGVLACYSLLVTVAFIIFWV',
    'tail_seq': 'RSKRSRLLHSDYMNMTPRRPGPTRKHYQPYAPPRDFAAYRSRDQRLPPDAHKPPGGGSFRTPIQEEQADAHSTLAKIRVKFSRSADAPAYQGGQNQLYNELNLGRREEYDVLDKRRGRDPEMGGKPRRKNPQEGLYNELQKDKMAEAYSEIGMKGERRRGKGHDCLYQGLSTATKDTYDALHMQALPPR'
}

test_example = {
    'peptide_signal_seq': 'MLLLVTSLLLCELPHPAFLLIP',
    'scfv_seq': 'DIQMTQSPSSLSASVGDRVTITCRASQSISSYLNWYQQKPGKAPKLLIYAASALQSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQAYYDPTTFGQGTKVEIKGSTSGSGKPGSGEGSTKGEVQLLESGGGLVQPGGSLRLSCAASGFTFSSYAMSWVRQAPGKGLEWVSTISTSGDYTTYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAKSTAYFDYWGQGTLVTVSS',
    'hinge_seq': 'AAATTTPAPRPPTPAPTIASQPLSLRPEACRPAAGGAVHTRGLDFACD',
    'tm_seq': 'FWVLVVVGGVLACYSLLVTVAFIIFWV',
    'tail_seq': 'RSKRSRLLHSDYMNMTPRRPGPTRKHYQPYAPPRDFAAYRSRKRGRKKLLYIFKQPFMRPVQTTQEEDGCSCRFPEEEEGGCELRVKFSRSADAPAYQQGQNQLYNELNLGRREEYDVLDKRRGRDPEMGGKPRRKNPQEGLYNELQKDKMAEAYSEIGMKGERRRGKGHDGLYQGLSTATKDTYDALHMQALPPR'
}

if 'Validation' in example_to_load:
    loaded_seqs = validation_example
else:
    loaded_seqs = test_example

peptide_signal_seq = loaded_seqs['peptide_signal_seq']
scfv_seq = loaded_seqs['scfv_seq']
hinge_seq = loaded_seqs['hinge_seq']
tm_seq = loaded_seqs['tm_seq']
tail_seq = loaded_seqs['tail_seq']

print(f"Example '{example_to_load}' loaded. Modify the fields below if you wish, then run the cell.")

Example 'Test Set Example (JAG1-F1)' loaded. Modify the fields below if you wish, then run the cell.


In [ ]:
# ==============================================================================
# Cell 4b: Manual Sequence Entry
# ==============================================================================
#@title Sub-option 4b: Manually Enter the CAR Sequences to Predict
#@markdown Fill in the fields below with the sequences of each domain, then run this cell to load them into memory.

peptide_signal_seq = "MLLLVTSLLLCELPHPAFLLIP" #@param {type:"string"}
scfv_seq = "DIQMTQSPSSLSASVGDRVTITCRASQSISSYLNWYQQKPGKAPKLLIYAASALQSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQAYYDPTTFGQGTKVEIKGSTSGSGKPGSGEGSTKGEVQLLESGGGLVQPGGSLRLSCAASGFTFSSYAMSWVRQAPGKGLEWVSTISTSGDYTTYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAKSTAYFDYWGQGTLVTVSS" #@param {type:"string"}
hinge_seq = "AAATTTPAPRPPTPAPTIASQPLSLRPEACRPAAGGAVHTRGLDFACD" #@param {type:"string"}
tm_seq = "FWVLVVVGGVLACYSLLVTVAFIIFWV" #@param {type:"string"}
tail_seq = "RSKRSRLLHSDYMNMTPRRPGPTRKHYQPYAPPRDFAAYRSRKRGRKKLLYIFKQPFMRPVQTTQEEDGCSCRFPEEEEGGCELRVKFSRSADAPAYQQGQNQLYNELNLGRREEYDVLDKRRGRDPEMGGKPRRKNPQEGLYNELQKDKMAEAYSEIGMKGERRRGKGHDGLYQGLSTATKDTYDALHMQALPPR" #@param {type:"string"}

print("✅ Sequences entered and stored in variables.")
print("You can now run 'Cell 5: Run the Prediction Pipeline' to obtain the result.")

✅ Sequences entered and stored in variables.
You can now run 'Cell 5: Run the Prediction Pipeline' to obtain the result.


### Step 5: Run the Prediction Pipeline

In [ ]:
# ==============================================================================
# Cell 5 (Complete and Corrected): Run the Single Prediction Pipeline
# ==============================================================================
print("Starting the prediction pipeline...")

# 1. Collect the sequences in a dictionary
domain_inputs = {
    'Peptide_Signal': peptide_signal_seq,
    'scFv': scfv_seq,
    'Hinge': hinge_seq,
    'TM': tm_seq,
    'Tail': tail_seq
}

# --- NEW VALIDATION BLOCK ---
print("Validating input sequences...")
all_validation_errors = []
cleaned_domain_inputs = {}
for domain_name, sequence in domain_inputs.items():
    cleaned_seq, errors = validate_and_clean_sequence(sequence, domain_name)
    if errors:
        all_validation_errors.extend(errors)
    cleaned_domain_inputs[domain_name] = cleaned_seq

# If errors are found, display them and stop the pipeline
if all_validation_errors:
    print("\n❌ ERROR: Invalid characters were detected in the sequences. Prediction cancelled.")
    for error in all_validation_errors:
        print(f"   - {error}")
    # Clear any previous result to avoid confusion
    if 'single_prediction_result' in locals():
        del single_prediction_result
# If everything is valid, continue with the prediction pipeline
else:
    print("✅ Valid sequences.")
    with np.errstate(all='ignore'): # Suppress calculation warnings on empty sequences
        # --- Start of the Prediction Pipeline ---

        # Use the cleaned sequences
        domain_inputs = cleaned_domain_inputs

        features_dict = {}
        protein_domains_internal = ['Peptide_Signal', 'scFv', 'Hinge', 'TM', 'Tail']

        # 2. Calculate the biophysical features
        for domain in protein_domains_internal:
            biophys_feats = calculate_prot_features(domain_inputs[domain])
            for fname, fvalue in biophys_feats.items():
                features_dict[f"biophys_{domain}_{fname}"] = fvalue

        # 3. Calculate the 'scFv_family' feature
        try:
            scfv_len = features_dict['biophys_scFv_length']
            if scfv_len <= 200: scfv_family = 'Short'
            elif scfv_len <= 300: scfv_family = 'Standard'
            else: scfv_family = 'Long'
        except KeyError:
            scfv_family = 'Standard' # Default value

        features_dict['family_Short'] = 1 if scfv_family == 'Short' else 0
        features_dict['family_Standard'] = 1 if scfv_family == 'Standard' else 0
        features_dict['family_Long'] = 1 if scfv_family == 'Long' else 0

        # 4. Calculate the ESM-2 embeddings
        print("Computing embeddings (may take ~30 seconds)...")
        for domain in protein_domains_internal:
            embedding = get_single_embedding(domain_inputs[domain], esm_model, tokenizer, device)
            for i, val in enumerate(embedding):
                features_dict[f"emb_{domain}_{i}"] = val

        # 5. Create the DataFrame
        X_new = pd.DataFrame([features_dict])

        # 6. Align the columns
        X_new_aligned = X_new.reindex(columns=model_columns, fill_value=0)

        # 7. Apply scaling
        print("Applying scaling...")
        X_new_scaled = scaler.transform(X_new_aligned)

        # SAFETY: Check that the final number of columns matches
        if X_new_aligned.shape[1] != final_model.n_features_in_:
             raise ValueError(f"Dimension error! The model expects {final_model.n_features_in_} features, but received {X_new_aligned.shape[1]}.")

        # 8. Prediction
        prediction_proba = final_model.predict_proba(X_new_scaled)[0]
        prediction = final_model.predict(X_new_scaled)[0]

        print("\n--- Prediction complete! ---")

        # Store the results for the display cell
        single_prediction_result = {'prediction': prediction, 'probability': prediction_proba[1]}

Starting the prediction pipeline...


NameError: name 'peptide_signal_seq' is not defined

### Step 6: Single Prediction Results

In [ ]:
# ==============================================================================
# Cell 6 (Complete and Corrected): Detailed Result Display
# ==============================================================================
from IPython.display import display, HTML

# Check whether a prediction result exists before trying to display it
try:
    if 'single_prediction_result' in locals():
        confidence_score = single_prediction_result['probability']
        prediction = single_prediction_result['prediction']

        # --- NEW BLOCK: SEQUENCE DISPLAY PREPARATION ---

        # Function to truncate long sequences for a clean display
        def truncate_sequence(seq, max_len=80):
            if len(seq) > max_len:
                return f"{seq[:35]}...{seq[-20:]}"
            return seq

        # Build the HTML block to display the sequences used
        sequences_html = f"""
        <div style="border: 1px solid #ccc; padding: 15px; border-radius: 8px; background-color: #f9f9f9; margin-bottom: 15px;">
            <h3 style="margin-top: 0;">Sequences used for this prediction:</h3>
            <ul style="font-family: monospace; list-style-type: none; padding-left: 0;">
                <li style="margin-bottom: 5px;"><strong>Peptide Signal:</strong> {truncate_sequence(peptide_signal_seq)}</li>
                <li style="margin-bottom: 5px;"><strong>scFv:</strong> {truncate_sequence(scfv_seq)}</li>
                <li style="margin-bottom: 5px;"><strong>Hinge:</strong> {truncate_sequence(hinge_seq)}</li>
                <li style="margin-bottom: 5px;"><strong>TM:</strong> {truncate_sequence(tm_seq)}</li>
                <li style="margin-bottom: 5px;"><strong>Tail:</strong> {truncate_sequence(tail_seq)}</li>
            </ul>
        </div>
        """

        # --- END OF NEW BLOCK ---

        # Build the HTML block for the prediction result
        if prediction == 1:
            result_html = f"""
            <div style="border: 2px solid green; padding: 20px; border-radius: 10px; background-color: #e8f5e9;">
                <h2 style="color: green; margin-top:0;">Prediction: High Cytotoxicity</h2>
                <p style="font-size: 1.2em;">The model is <strong>{confidence_score:.2%}</strong> confident that this CAR will be effective.</p>
            </div>
            """
        else:
            result_html = f"""
            <div style="border: 2px solid orange; padding: 20px; border-radius: 10px; background-color: #fff3e0;">
                <h2 style="color: orange; margin-top:0;">Prediction: Low Cytotoxicity</h2>
                <p style="font-size: 1.2em;">The model is <strong>{1-confidence_score:.2%}</strong> confident that this CAR will be ineffective ('High' score: {confidence_score:.2%}).</p>
            </div>
            """

        # Combined display of sequences and result
        display(HTML(sequences_html + result_html))

    else:
        # Handles the case where cell 5 was run but failed validation
        # (the error message is already displayed by cell 5)
        pass

except NameError:
    # Handles the case where cell 5 was never run
    display(HTML("<div style='border: 1px solid #ccc; padding: 10px; border-radius: 5px; color: #555;'>No result to display. Please first run Cell 5 to generate a prediction.</div>"))

---
### **Option B: Batch Prediction (Excel File)**
---

### Step 7: Predict on a Full Excel File

If you have multiple constructs to predict, run the cell below to upload an Excel file.

**Required format:**
*   The file must be in `.xlsx` or `.xls` format.
*   The column names for the sequences **must be exactly**: `Peptide_Signal_(Protein)`, `scFv_(Protein)`, `Hinge_(Protein)`, `TM_(Protein)`, `Tail_(Protein)`.
*   A `Construct ID` column is recommended to identify your sequences.

In [4]:
# ==============================================================================
# Cell 7 (Complete and Corrected Version): Predict on an Excel File
# ==============================================================================
from google.colab import files
from tqdm.auto import tqdm
import io

print("Please upload your Excel file containing the sequences to predict.")
uploaded_excel = files.upload()

if not uploaded_excel:
    print("No file was uploaded. Operation cancelled.")
else:
    filename = next(iter(uploaded_excel))
    content = uploaded_excel[filename]

    try:
        df_to_predict = pd.read_excel(io.BytesIO(content))
        print(f"File '{filename}' loaded successfully. {len(df_to_predict)} constructs to predict.")

        # --- Start of the Batch Prediction Pipeline ---
        all_rows_features = []
        validation_errors_list = []

        protein_domains_original_names = ['Peptide_Signal_(Protein)', 'AntigenBindingDomain_(Protein)', 'Hinge_(Protein)', 'TM_(Protein)', 'Tail_(Protein)']
        protein_domains_internal = ['Peptide_Signal', 'AntigenBindingDomain', 'Hinge', 'TM', 'Tail']
        rename_map = {orig: clean for orig, clean in zip(protein_domains_original_names, protein_domains_internal)}

        df_processed = df_to_predict.rename(columns=rename_map)

        progress_bar = tqdm(df_processed.iterrows(), total=len(df_processed), desc="Processing constructs")

        for index, row in progress_bar:
            row_features = {}
            row_errors = []

            # --- PER-ROW VALIDATION BLOCK ---
            cleaned_row_sequences = {}
            for domain in protein_domains_internal:
                original_sequence = row.get(domain)
                cleaned_seq, errors = validate_and_clean_sequence(original_sequence, domain)
                if errors:
                    row_errors.extend(errors)
                cleaned_row_sequences[domain] = cleaned_seq

            validation_status = ' | '.join(row_errors) if row_errors else 'OK'
            validation_errors_list.append(validation_status)

            # If the row has a sequence error, we still create "empty" features
            # but they will not be used for the final prediction of this row
            if validation_status != 'OK':
                all_rows_features.append({}) # Empty dictionary as a marker
                continue
            # --- END OF VALIDATION BLOCK ---

            with np.errstate(all='ignore'):
                # Compute biophysical features
                for domain in protein_domains_internal:
                    biophys_feats = calculate_prot_features(cleaned_row_sequences[domain])
                    for fname, fvalue in biophys_feats.items():
                        row_features[f"biophys_{domain}_{fname}"] = fvalue

                # Compute the 'scFv_family' feature
                try:
                    scfv_len = row_features['biophys_AntigenBindingDomain_length']
                    if scfv_len <= 200: scfv_family = 'Short'
                    elif scfv_len <= 300: scfv_family = 'Standard'
                    else: scfv_family = 'Long'
                except KeyError:
                    scfv_family = 'Standard'

                row_features['family_Short'] = 1 if scfv_family == 'Short' else 0
                row_features['family_Standard'] = 1 if scfv_family == 'Standard' else 0
                row_features['family_Long'] = 1 if scfv_family == 'Long' else 0

                # Compute the ESM-2 embeddings
                for domain in protein_domains_internal:
                    embedding = get_single_embedding(cleaned_row_sequences[domain], esm_model, tokenizer, device)
                    for i, val in enumerate(embedding):
                        row_features[f"emb_{domain}_{i}"] = val

            all_rows_features.append(row_features)

        # --- Build the final DataFrame and run predictions ---
        # Filter out rows with errors before creating the DataFrame for the model
        valid_rows_indices = [i for i, status in enumerate(validation_errors_list) if status == 'OK']
        valid_rows_features = [all_rows_features[i] for i in valid_rows_indices]

        if valid_rows_features:
            X_new_batch = pd.DataFrame(valid_rows_features)
            X_new_aligned = X_new_batch.reindex(columns=model_columns, fill_value=0)

            print("\nApplying scaling to the batch of valid data...")
            X_new_scaled_batch = scaler.transform(X_new_aligned)

            final_predictions_proba = final_model.predict_proba(X_new_scaled_batch)
            final_predictions = final_model.predict(X_new_scaled_batch)

            confidences = final_predictions_proba[:, 1]
            predictions = ['High' if pred == 1 else 'Low' for pred in final_predictions]
        else:
            print("\nNo valid rows found for prediction.")
            predictions = []
            confidences = []

        # Add the results to the original DataFrame
        df_to_predict['Validation_Status'] = validation_errors_list
        # Initialize the prediction columns with default values
        df_to_predict['Predicted_Cytotoxicity'] = 'Validation Error'
        df_to_predict['Confidence_Prediction_High'] = np.nan
        # Fill in predictions only for valid rows
        df_to_predict.loc[valid_rows_indices, 'Predicted_Cytotoxicity'] = predictions
        df_to_predict.loc[valid_rows_indices, 'Confidence_Prediction_High'] = confidences

        # Export the result
        output_filename = filename.replace('.xlsx', '').replace('.xls', '') + '_predictions.xlsx'
        with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:
            df_to_predict.to_excel(writer, index=False, sheet_name='Predictions')
            workbook  = writer.book
            worksheet = writer.sheets['Predictions']
            percent_format = workbook.add_format({'num_format': '0.00%'})
            conf_col_idx = df_to_predict.columns.get_loc('Confidence_Prediction_High')
            worksheet.set_column(conf_col_idx, conf_col_idx, 25, percent_format)

        print(f"\n✅ Predictions complete. The file '{output_filename}' has been generated.")
        print("It contains a 'Validation_Status' column indicating any errors.")

    except Exception as e:
        print(f"\n❌ An error occurred while processing the Excel file: {e}")
        print("Please check that the file format and column names are correct.")

Please upload your Excel file containing the sequences to predict.


Saving dataset_inference_validation.xlsx to dataset_inference_validation.xlsx
File 'dataset_inference_validation.xlsx' loaded successfully. 10 constructs to predict.


Processing constructs:   0%|          | 0/10 [00:00<?, ?it/s]


Applying scaling to the batch of valid data...

✅ Predictions complete. The file 'dataset_inference_validation_predictions.xlsx' has been generated.
It contains a 'Validation_Status' column indicating any errors.


Cell 8 will take the df_to_predict results DataFrame (created and populated in Cell 7) and format it for a styled display, similar to what Cell 27 of the training notebook does.

Note that here, we do not have a "ground truth" or notion of "correct prediction", since this is new data. The table will therefore focus on a clear display of the predictions and confidence scores.


### **Step 8:** Visualization of Batch Prediction Results

In [5]:
# ==============================================================================
# Cell 8 (Complete and Corrected Version): Batch Results Visualization
# ==============================================================================
from IPython.display import display, HTML

# Check whether the 'df_to_predict' prediction DataFrame was generated by the previous cell
try:
    if 'df_to_predict' in locals() and not df_to_predict.empty:
        print("\n--- Summary Table of Predictions on the Excel File ---")

        # --- NEW BLOCK: TRUNCATION FUNCTION ---
        def truncate_sequence_for_display(seq, max_len=60):
            """Truncates a sequence for display in the DataFrame."""
            if isinstance(seq, str) and len(seq) > max_len:
                return f"{seq[:25]}...{seq[-15:]}"
            return seq
        # --- END OF NEW BLOCK ---

        # Select the relevant columns for display
        display_cols = [
            'Construct ID',
            'Validation_Status',
            'Predicted_Cytotoxicity',
            'Confidence_Prediction_High'
        ]

        sequence_cols_to_display = ['scFv_(Protein)', 'Hinge_(Protein)']
        for col in sequence_cols_to_display:
            if col in df_to_predict.columns:
                display_cols.append(col)

        display_cols = [col for col in display_cols if col in df_to_predict.columns]

        # Create a copy for formatting
        results_to_style = df_to_predict[display_cols].copy()

        # --- APPLY TRUNCATION TO SEQUENCE COLUMNS ---
        for col in sequence_cols_to_display:
            if col in results_to_style.columns:
                results_to_style[col] = results_to_style[col].apply(truncate_sequence_for_display)
        # --- END OF APPLICATION ---

        # Apply conditional styling for better readability
        def style_predictions_and_errors(df):
            style = pd.DataFrame('', index=df.index, columns=df.columns)
            mask_error = df['Validation_Status'] != 'OK'
            style.loc[mask_error, :] = 'background-color: #f2f2f2; color: #888'

            mask_high = (df['Predicted_Cytotoxicity'] == 'High') & (~mask_error)
            mask_low = (df['Predicted_Cytotoxicity'] == 'Low') & (~mask_error)

            style.loc[mask_high, :] = 'background-color: #e8f5e9'
            style.loc[mask_low, :] = 'background-color: #fff3e0'
            return style

        # Apply formatting and styling
        styled_df = results_to_style.style.format({
            'Confidence_Prediction_High': '{:.2%}'
        }, na_rep="-").apply(style_predictions_and_errors, axis=None)

        display(styled_df)

    else:
        print("No batch prediction results to display. Please first run Cell 7 with an Excel file.")

except NameError:
    print("No batch prediction results to display. Please first run Cell 7 with an Excel file.")


--- Summary Table of Predictions on the Excel File ---


,Construct ID,Validation_Status,Predicted_Cytotoxicity,Confidence_Prediction_High,Hinge_(Protein)
0,sdHER2-5_KIRS2DAP12,OK,Low,49.61%,AIEQKLISEEDLAIGSNSSDPLLVSVTGNPSNSWPSPTEPSSKTGNPRHLH
1,F265_KIRS2DAP12,OK,High,50.17%,AIEQKLISEEDLAIGSNSSDPLLVSVTGNPSNSWPSPTEPSSKTGNPRHLH
2,scFv2_EH2_Myc_KIRS2,OK,High,50.78%,SNSSDPLLVSVTGNPSNSWPSPTEPSSKTGNPRHLH
3,scFv1_EH2_Myc_KIRS2,OK,High,50.78%,SNSSDPLLVSVTGNPSNSWPSPTEPSSKTGNPRHLH
4,M410_BBz,OK,Low,49.30%,VSGTIEVMYPPPYLDNEKSNGTIIHVKGKHLCPSPLFPGPSKP
5,14g2a_KIRS2_DNAM1,OK,Low,49.19%,AIGSNSSDPLLVSVTGNPSNSWPSPTEPSSKTGNPRHLH
6,pHE_B7H3_28z_mKate2,OK,High,50.17%,IEVMYPPPYLDNEKSNGTIIHVKGKHLCPSPLFPGPSKP
7,pHUS 14g2a-Myc-KIRS2-dIL2RB-YXXQ BSD,OK,Low,49.19%,AIGSNSSDPLLVSVTGNPSNSWPSPTEPSSKTGNPRHLH
8,pHRSIN_RWG8_BBz,OK,High,50.37%,TTTPAPRPPTPAPTIASQPLSLRPEACRPAAGGAVHTRGLDFACD
9,pHRSIN_cita-cel_BCMA_BBz,OK,Low,49.65%,TSTTTPAPRPPTPAPTIASQPLSLRPEACRPAAGGAVHTRGLDFACD
